Code based on Svet's notebook

In [39]:
import glob
import gc
import cudf
from numba import cuda
import pandas as pd
from tqdm import tqdm
import os

os.chdir('/home/rapids/Capstone')


In [40]:
# Function to convert a CSV file to a Parquet file
def convert_csv_to_parquet(file_path):
    # Read the CSV file
    df = cudf.read_csv(file_path)

    # Construct the new file path for the Parquet file
    parquet_file_path = file_path.replace('.csv', '.parquet')

    # Write the DataFrame to a Parquet file
    df.to_parquet(parquet_file_path, index=False)

    # Delete the DataFrame to free up memory
    del df

# Get the list of CSV files
file_list = glob.glob('data/*.csv')
print(f"Found files: {file_list}")

# Process files with a progress bar
for file_path in tqdm(file_list, desc="Converting CSV to Parquet"):
    convert_csv_to_parquet(file_path)

    
    

Found files: ['data/2023_01_Gener_BicingNou_ESTACIONS.csv', 'data/2021_10_Octubre_BicingNou_ESTACIONS.csv', 'data/2024_04_Abril_BicingNou_ESTACIONS.csv', 'data/2024_03_Marc_BicingNou_ESTACIONS.csv', 'data/2022_01_Gener_BicingNou_ESTACIONS.csv', 'data/2023_11_Novembre_BicingNou_ESTACIONS.csv', 'data/2022_12_Desembre_BicingNou_ESTACIONS.csv', 'data/2024_02_Febrer_BicingNou_ESTACIONS.csv', 'data/2022_05_Maig_BicingNou_ESTACIONS.csv', 'data/2022_10_Octubre_BicingNou_ESTACIONS.csv', 'data/2024_05_Maig_BicingNou_ESTACIONS.csv', 'data/2020_01_Gener_BicingNou_ESTACIONS.csv', 'data/2022_09_Setembre_BicingNou_ESTACIONS.csv', 'data/2023_07_Juliol_BicingNou_ESTACIONS.csv', 'data/2023_08_Agost_BicingNou_ESTACIONS.csv', 'data/2021_11_Novembre_BicingNou_ESTACIONS.csv', 'data/2021_12_Desembre_BicingNou_ESTACIONS.csv', 'data/2022_04_Abril_BicingNou_ESTACIONS.csv', 'data/2020_08_Agost_BicingNou_ESTACIONS.csv', 'data/2023_02_Febrer_BicingNou_ESTACIONS.csv', 'data/2020_05_Maig_BicingNou_ESTACIONS.csv', 'd

Converting CSV to Parquet: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:26<00:00,  2.02it/s]


In [41]:
# For each parquet file, read it and, if present, delete the columns 'traffic', 'V1' and 'last_updated'
def clean_parquet_files():
    # For each parquet file, read it and, if present, delete the columns 'traffic', 'V1' and 'last_updated'
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Cleaning Parquet Files"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Drop the columns if they exist
        columns_to_drop = ['traffic', 'V1', 'last_updated']
        for column in columns_to_drop:
            if column in df.columns:
                df.drop(column, axis=1, inplace=True)

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
clean_parquet_files()

Cleaning Parquet Files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:13<00:00,  3.94it/s]


In [42]:
# For every parquet file, replace any negative value in the numeric columns with zero


def replace_negative_values_with_zero():
    # For every parquet file, replace any negative value in the numeric columns with zero
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Replacing Negative Values"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Replace negative values with zero
        for column in df.columns:
            if df[column].dtype in ['int8', 'int16', 'int32', 'int64', 'float32', 'float64']:
                df[column] = df[column].clip(lower=0)

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
replace_negative_values_with_zero()

Replacing Negative Values: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:14<00:00,  3.74it/s]


In [43]:
# Convert the 'last_reported' column from Unix timestamp to datetime. After that, sort the dataframe in ascending order. For each year, day and hour, create a new column with the corresponding value. Finally, reset the index of the DataFrame and write it back to the Parquet file.

def convert_last_reported_to_datetime():
    # Convert the 'last_reported' column from Unix timestamp to datetime
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Converting 'last_reported' to Datetime"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Convert the 'last_reported' column from Unix timestamp to datetime
        df['last_reported'] = cudf.to_datetime(df['last_reported'], unit='s')

        # Sort the DataFrame in ascending order
        df = df.sort_values('last_reported')

        # Create new columns for year, month, day and hour
        df['year'] = df['last_reported'].dt.year
        df['month'] = df['last_reported'].dt.month
        df['day'] = df['last_reported'].dt.day
        df['hour'] = df['last_reported'].dt.hour

        # Reset the index of the DataFrame
        df.reset_index(drop=True, inplace=True)

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
convert_last_reported_to_datetime()

Converting 'last_reported' to Datetime:   2%|█▋                                                                                        | 1/53 [00:00<00:14,  3.51it/s]

Converting 'last_reported' to Datetime: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:17<00:00,  3.00it/s]


In [44]:
# Delete the last reported column

def delete_last_reported_column():
    # Delete the last reported column
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Deleting 'last_reported' Column"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Drop the last reported column
        df.drop('last_reported', axis=1, inplace=True)

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
delete_last_reported_column()

Deleting 'last_reported' Column: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:09<00:00,  5.41it/s]


In [45]:
# For each parque file, for each year, day and hour, calculate the mode of columns 'station_id', 'is_installed', 'is_renting', 'is_returning' and 'is_charging_station'. For the rest of the columns calculate the average. Write the resulting DataFrame back to the Parquet file.

def calculate_aggregates():
    # For each parquet file, import it to a dataframe
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Average of values per hour"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Create a dataframe without status and is_charging_station
        dfa = df.drop(['status', 'is_charging_station'], axis=1)
        # Group by year, month, day and hour and station_id and calculate average
        dfa = dfa.groupby(['year', 'month', 'day', 'hour', 'station_id']).mean().reset_index()

        # Create a dataframe with only status and is_charging_station
        dfb = df[['year', 'month', 'day', 'hour', 'station_id', 'status', 'is_charging_station']]
        # Encode the status and is_charging_station columns and make them numerical
        # print(f"Encoding 'status' column: {dfb['status'].unique()} to numerical categories")
        dfb['status'] = dfb['status'].astype('category').cat.codes
        # print(f"Encoding 'is_charging_station' column: {dfb['is_charging_station'].unique()} to numerical categories")
        dfb['is_charging_station'] = dfb['is_charging_station'].astype('category').cat.codes
        
        # Group by year, day and hour and station_id and calculate mean
        dfb = dfb.groupby(['year', 'month', 'day', 'hour', 'station_id']).mean().reset_index()
        # In the status and is_charging_station columns, round the values to the nearest integer
        dfb['status'] = dfb['status'].round().astype('int')
        dfb['is_charging_station'] = dfb['is_charging_station'].round().astype('int')

        # Merge the two dataframes
        df = cudf.merge(dfa, dfb, on=['year', 'month', 'day', 'hour', 'station_id'])

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
calculate_aggregates()

Average of values per hour:   4%|███▊                                                                                                  | 2/53 [00:00<00:09,  5.21it/s]

Average of values per hour: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:10<00:00,  5.06it/s]


In [46]:
# Define excel file
excel_file = 'Informacio_Estacions_Bicing_2025.xlsx'

# Load excel file
excel_df = pd.read_excel(excel_file)
# Ensure 'station_id' and 'capacity' are integers
excel_df['station_id'] = excel_df['station_id'].astype('int')
excel_df['capacity'] = excel_df['capacity'].astype('int')

# Convert the DataFrame to a cuDF DataFrame
excel_df = cudf.DataFrame.from_pandas(excel_df)

# Add the information from the capacity excel to each parquet file
def add_capacity_info():
    # For each parquet file, import it to a dataframe
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Adding Capacity Information"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Merge the two dataframes
        df = cudf.merge(df, excel_df, on='station_id', how='left')

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df
        
add_capacity_info()
# Load excel file
# df = pd.read_excel(excel_file)

# # Ensure 'station_id' and 'capacity' are integers
# df['station_id'] = df['station_id'].astype('int')
# df['capacity'] = df['capacity'].astype('int')

# df = cudf.DataFrame.from_pandas(df)

# dfa = cudf.read_parquet('data/2022_11_Novembre_BicingNou_ESTACIONS.parquet')

# # Merge the two dataframes
# df = cudf.merge(dfa, df, on='station_id', how='left')
# df.head(5)

Adding Capacity Information:   4%|███▊                                                                                                 | 2/53 [00:00<00:05,  8.93it/s]

Adding Capacity Information: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:05<00:00, 10.53it/s]


In [47]:
# For every parquet file, calculate the percentage of docks available and write the resulting DataFrame back to the Parquet file.

def calculate_docks_available():
    # For every parquet file, calculate the percentage of docks available
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Calculating Docks Available"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Calculate the percentage of docks available
        df['docks_available'] = df['num_docks_available'] / df['capacity']

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
calculate_docks_available()

Calculating Docks Available: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:04<00:00, 10.80it/s]


In [48]:
# There might be some situations in which the 'docks_available' column has a value greater than 1. In these cases, the value should be set to 1.

def set_docks_available_to_1():
    # For every parquet file, set the 'docks_available' column to 1 if it is greater than 1
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Setting 'docks_available' to 1"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Set the 'docks_available' column to 100 if it is greater than 100
        df['docks_available'] = df['docks_available'].clip(upper=1)

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
set_docks_available_to_1()

Setting 'docks_available' to 1: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:05<00:00,  9.85it/s]


In [49]:
# INSTALL THE PIPS IF YOU DO NOT HAVE THEM (I installed them in my virtual environment in Linux and do not need them in my code)
# pip install openmeteo-requests
# pip install requests-cache retry-requests numpy pandas


import openmeteo_requests

import requests_cache
import pandas as pd
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 41.3888,
	"longitude": 2.159,
	"start_date": "2020-01-01",
	"end_date": "2024-12-31",
	"hourly": ["temperature_2m", "rain", "apparent_temperature", "relative_humidity_2m", "wind_speed_10m", "is_day", "sunshine_duration"],
	"timezone": "Europe/Berlin"
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation {response.Elevation()} m asl")
print(f"Timezone {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

							# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_rain = hourly.Variables(1).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(2).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(3).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()
hourly_is_day = hourly.Variables(5).ValuesAsNumpy()
hourly_sunshine_duration = hourly.Variables(6).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["rain"] = hourly_rain
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["sunshine_duration"] = hourly_sunshine_duration

hourly_dataframe = pd.DataFrame(data = hourly_data)
print(hourly_dataframe)

Coordinates 41.37082290649414°N 2.068965435028076°E
Elevation 44.0 m asl
Timezone b'Europe/Berlin'b'GMT+1'
Timezone difference to GMT+0 3600 s
                           date  temperature_2m  rain  apparent_temperature  \
0     2019-12-31 23:00:00+00:00            4.64   0.0              1.536496   
1     2020-01-01 00:00:00+00:00            4.24   0.0              0.874873   
2     2020-01-01 01:00:00+00:00            3.69   0.0              0.518975   
3     2020-01-01 02:00:00+00:00            3.29   0.0              0.066232   
4     2020-01-01 03:00:00+00:00            2.79   0.0             -0.455314   
...                         ...             ...   ...                   ...   
43843 2024-12-31 18:00:00+00:00            8.89   0.0              8.038103   
43844 2024-12-31 19:00:00+00:00            8.24   0.0              6.649469   
43845 2024-12-31 20:00:00+00:00            7.64   0.0              5.839178   
43846 2024-12-31 21:00:00+00:00            7.54   0.0              

In [53]:
# Correctly generate the datetime range explicitly from hourly.Time()
hourly_dataframe["date"] = pd.date_range(
    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
    periods=len(hourly_temperature_2m),
    freq=pd.Timedelta(seconds=hourly.Interval())
).tz_convert("Europe/Berlin")

# Add weather variables to the dataframe
hourly_dataframe["temperature_2m"] = hourly_temperature_2m
hourly_dataframe["rain"] = hourly_rain
hourly_dataframe["apparent_temperature"] = hourly_apparent_temperature
hourly_dataframe["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_dataframe["wind_speed_10m"] = hourly_wind_speed_10m
hourly_dataframe["sunshine_duration"] = hourly_sunshine_duration

# Extract new date and hour columns explicitly
hourly_dataframe["year"] = hourly_dataframe["date"].dt.year.astype('int32')
hourly_dataframe["month"] = hourly_dataframe["date"].dt.month.astype('int32')
hourly_dataframe["day"] = hourly_dataframe["date"].dt.day.astype('int32')
hourly_dataframe["hour"] = hourly_dataframe["date"].dt.hour.astype('int32')
# Drop the date column
hourly_dataframe.drop(columns=['date'], inplace=True)
# Drop duplicates
hourly_dataframe.drop_duplicates(inplace=True)

# Verify corrected extraction
print("✅ Corrected extraction sample:")
print(hourly_dataframe[['hour']].head(10))

# Clearly confirm hours
print("\n✅ Unique hours after timezone correction:")
print(sorted(hourly_dataframe['hour'].unique()))


display(hourly_dataframe.head(5))


✅ Corrected extraction sample:
   hour
0     0
1     1
2     2
3     3
4     4
5     5
6     6
7     7
8     8
9     9

✅ Unique hours after timezone correction:
[np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]


,temperature_2m,rain,apparent_temperature,relative_humidity_2m,wind_speed_10m,sunshine_duration,year,month,day,hour
0,4.64,0.0,1.536496,85.641792,9.504273,0.0,2020,1,1,0
1,4.24,0.0,0.874873,87.136894,11.113451,0.0,2020,1,1,1
2,3.69,0.0,0.518975,89.924957,9.659814,0.0,2020,1,1,2
3,3.29,0.0,0.066232,90.538269,9.659814,0.0,2020,1,1,3
4,2.79,0.0,-0.455314,92.136627,9.511088,0.0,2020,1,1,4


In [59]:
# For every parquet file, merge the dataframe with hourtly_dataframe in the rows with the same year, month, day and hour. Write the resulting DataFrame back to the Parquet file.

cu_hourly_dataframe = cudf.DataFrame.from_pandas(hourly_dataframe)

def merge_weather_data():
    # For every parquet file, merge the dataframe with weather data
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Merging Weather Data"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Merge the two dataframes
        df = cudf.merge(df, cu_hourly_dataframe, on=['year', 'month', 'day', 'hour'], how='left')

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
merge_weather_data()

Merging Weather Data:   2%|██                                                                                                          | 1/53 [00:00<00:09,  5.40it/s]

Merging Weather Data: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:06<00:00,  7.75it/s]


In [ ]:
import sqlite3

# Define the SQLite database file
sqlite_file = 'data.sqlite'

def save_parquet_to_sqlite():
    # Connect to the SQLite database (it will be created if it doesn't exist)
    conn = sqlite3.connect(sqlite_file)
    
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Saving Parquet to SQLite"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)
        
        # Convert cuDF DataFrame to pandas DataFrame for SQLite compatibility
        pandas_df = df.to_pandas()
        
        # Append the data to the SQLite table
        pandas_df.to_sql('data_table', conn, if_exists='append', index=False)
        
        # Delete the DataFrame to free up memory
        del df, pandas_df
    
    # Close the SQLite connection
    conn.close()

# Call the function
save_parquet_to_sqlite()


Combining Parquet Files:  36%|█████████████████████████████████████▎                                                                  | 19/53 [00:00<00:00, 35.33it/s]


MemoryError: std::bad_alloc: out_of_memory: CUDA error at: /opt/conda/include/rmm/mr/device/cuda_memory_resource.hpp

In [64]:
if 'df' in locals():
    del df
# del grouped
# del average_df
gc.collect()

cuda.current_context().deallocations.clear()
# cuda.select_device(0)
# cuda.close()